## Media de varias estaciones

In [1]:
"""
Calcular medias por fecha para varias columnas a partir de una lista de archivos CSV.
Comportamiento principal
  1) solo se procesan archivos que contienen la columna temporal datetime
  2) se agrupan filas con el mismo datetime dentro de cada archivo
     aplicando por defecto la media para resolver duplicados
  3) se concatenan las series alineando por datetime y se calcula la media
  4) el archivo de salida contiene unicamente las columnas <col>_mean
Requisitos
  pandas
"""

from pathlib import Path
import os
import pandas as pd
from typing import List

def compute_means_from_filelist(
    file_paths: List[str],
    columns: List[str],
    output_path: str = "media.csv",
    time_column: str = "datetime",
    parse_dates: bool = True,
    required_min_files: int = 1,
    duplicate_agg: str = "mean"  # opciones: mean first sum
) -> pd.DataFrame:
    """
    file_paths: lista de rutas a los CSV a procesar
    columns: lista de nombres de columna para los que se calculara la media
    output_path: ruta del archivo CSV de salida que contendra solo las medias
    time_column: nombre exacto de la columna temporal en los CSV
    parse_dates: si True intenta convertir time_column a tipo fecha
    required_min_files: numero minimo de archivos validos necesarios
    duplicate_agg: como resolver duplicados de datetime dentro de cada archivo
                   valores posibles: "mean" "first" "sum"
    """
    resolved = []
    for p in file_paths:
        ppath = Path(p).expanduser()
        if not ppath.exists():
            print(f"Aviso archivo no encontrado {p}")
            continue
        resolved.append(str(ppath))

    if not resolved:
        raise FileNotFoundError("No se localizaron archivos validos en la lista proporcionada")

    per_variable_series = {col: [] for col in columns}
    valid_file_count = 0

    for fpath in resolved:
        try:
            df = pd.read_csv(fpath)
        except Exception as e:
            print(f"Aviso fallo lectura {fpath} Ignorando este archivo. Error: {e}")
            continue

        if time_column not in df.columns:
            print(f"Aviso {os.path.basename(fpath)} no contiene la columna temporal {time_column}. Archivo omitido.")
            continue

        if parse_dates:
            df[time_column] = pd.to_datetime(df[time_column], errors="coerce")
        df = df.dropna(subset=[time_column])
        if df.shape[0] == 0:
            print(f"Aviso {os.path.basename(fpath)} no tiene filas con {time_column} valido. Archivo omitido.")
            continue

        df = df.set_index(time_column)

        basename = os.path.basename(fpath)
        found_any_column = False
        for col in columns:
            if col in df.columns:
                s = pd.to_numeric(df[col], errors="coerce")
                # si el indice contiene duplicados, agrupar por indice y agregar
                if s.index.has_duplicates:
                    if duplicate_agg == "mean":
                        s = s.groupby(level=0).mean()
                    elif duplicate_agg == "first":
                        s = s[~s.index.duplicated(keep="first")]
                    elif duplicate_agg == "sum":
                        s = s.groupby(level=0).sum()
                    else:
                        # si el parametro no es valido usar mean
                        s = s.groupby(level=0).mean()
                # eliminar series vacias tras conversion
                if s.dropna().shape[0] == 0:
                    # no hay datos numericos utiles en esta columna
                    continue
                s.name = f"{basename}__{col}"
                per_variable_series[col].append(s)
                found_any_column = True

        if found_any_column:
            valid_file_count += 1
        else:
            print(f"Aviso {basename} no contiene ninguna de las columnas solicitadas {columns}. Archivo omitido.")

    if valid_file_count < required_min_files:
        raise ValueError(f"Solo {valid_file_count} archivos validos. Se requieren al menos {required_min_files}.")

    mean_frames = []
    for col, series_list in per_variable_series.items():
        if not series_list:
            print(f"Aviso no se encontraron datos para la variable {col} en los archivos procesados")
            continue
        # concatenar series alineando por indice datetime
        aligned = pd.concat(series_list, axis=1, join="outer")
        mean_series = aligned.mean(axis=1, skipna=True)
        mean_series.name = f"{col}_mean"
        mean_frames.append(mean_series)

    if not mean_frames:
        raise ValueError("No se pudieron calcular medias para ninguna de las columnas solicitadas")

    final_df = pd.concat(mean_frames, axis=1, join="outer")

    out_path = Path(output_path).expanduser()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    final_df.to_csv(out_path, index=True)
    print(f"Salida guardada en {out_path} Filas {final_df.shape[0]} Columnas {final_df.shape[1]}")

    return final_df

# Ejemplo de uso
if __name__ == "__main__":
    archivos = [
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/df_semicompletos/df__S_CEA_completo.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/df_semicompletos/df__S_Nord_completo.csv",
    "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/df_semicompletos/df__S_Port_completo.csv"
    ]
    columnas_a_promediar = ["NO", "NO2", "NOx", "O3", "Veloc.", "Direc.", "Temp.", "R.Sol.", "Pres."]
    # Cambie la ruta de salida por la que prefiera antes de ejecutar
    ruta_salida = "/Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/df_media_semicompletos/df_Sagunto_Media.csv"

    df_result = compute_means_from_filelist(
        file_paths=archivos,
        columns=columnas_a_promediar,
        output_path=ruta_salida,
        time_column="datetime",
        parse_dates=True,
        required_min_files=1
    )


Salida guardada en /Users/benjamincarbonell/Documents/GitHub/TFGFinal/Datos TFG/df_media_semicompletos/df_Sagunto_Media.csv Filas 122706 Columnas 9
